# 01 · Criação do Ambiente e Ingestão Bronze (V-Credit)

A ideia desse notebook é:
1. **Configurar o ambiente:** criar o catálogo, os schemas da arquitetura medalhão (`bronze`, `silver`, `gold`) e o volume onde ficam os CSVs;
2. **Ingerir as bases dimensionais:** carregar `base_atendentes.csv` e `base_motivos.csv` direto pra tabelas Delta Lake.

## 1. Parâmetros do ambiente


In [0]:
catalog = "vcredit"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"landing_path: {landing_path}")

## 2. Criação do catálogo, schemas e Volume


In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.Landing")

print("Catalog, schemas e landing_path criados")

## 3. Leitura dos CSVs


In [0]:
from pyspark.sql.functions import current_timestamp

path_base_atendentes = f"{landing_path}/base_atendentes.csv"
path_base_motivos = f"{landing_path}/base_motivos.csv"
path_canais = f"{landing_path}/canais.csv"
path_chamados = f"{landing_path}/chamados.csv"
path_chamados_hora = f"{landing_path}/Chamados_Hora.CSV"
path_clientes = f"{landing_path}/clientes.csv"
path_custos = f"{landing_path}/custos.csv"
path_pesquisa_satisfacao = f"{landing_path}/pesquisa_satisfacao.csv"

df_base_atendentes_raw = spark.read.csv(path_base_atendentes, header=True, inferSchema=False)
df_base_motivos_raw = spark.read.csv(path_base_motivos, header=True, inferSchema=False)
df_canais_raw = spark.read.csv(path_canais, header=True, inferSchema=False)
df_chamados_raw = spark.read.csv(path_chamados, header=True, inferSchema=False)
df_chamados_hora_raw = spark.read.csv(path_chamados_hora, header=True, inferSchema=False)
df_clientes_raw = spark.read.csv(path_clientes, header=True, inferSchema=False)
df_custos_raw = spark.read.csv(path_custos, header=True, inferSchema=False)
df_pesquisa_satisfacao_raw = spark.read.csv(path_pesquisa_satisfacao, header=True, inferSchema=False)

## 4. Ingestão na Camada Bronze (Delta Lake)

Puxando os arquivos CSV sem alterar os dados, adicionando o timestamp de auditoria e salvando em Delta Lake com `mode('overwrite')` para garantir idempotência.

In [0]:
from pyspark.sql.functions import current_timestamp

df_base_atendentes_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{catalog}.bronze.tb_atendentes")

df_base_motivos_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{catalog}.bronze.tb_motivos")

df_canais_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{catalog}.bronze.tb_canais")

df_chamados_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{catalog}.bronze.tb_chamados")

df_chamados_hora_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{catalog}.bronze.tb_chamados_hora")

df_clientes_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{catalog}.bronze.tb_clientes")

df_custos_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{catalog}.bronze.tb_custos")

df_pesquisa_satisfacao_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{catalog}.bronze.tb_pesquisa_satisfacao")

print("Tabelas Bronze gravadas com sucesso!")
display(spark.table(f"{catalog}.bronze.tb_chamados").limit(5))